# Station continuity breaks data preparation

Ustaw `YEAR` i przygotuj pliki roczne w `data_preparation_eksperyment.ipynb`. Dla pierwszego przejazdu każdego roweru szukamy jego ostatniego przejazdu we wcześniejszych latach. Jeśli w dostępnej historii nie ma poprzednika, pola `previous_end_*` pozostają puste. Zapis obejmuje przejazdy rozpoczęte w wybranym roku.


In [52]:
from pathlib import Path
import pandas as pd

In [53]:
YEAR = 2025


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "processed_data").is_dir():
            return path
    raise FileNotFoundError("Could not find project root containing processed_data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "processed_data" / "ready_2022_2025"


def annual_trip_file(year):
    path = DATA_DIR / f"final_trip_data_{year}.parquet"
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run data_preparation_eksperyment.ipynb with YEAR = {year} first."
        )
    return path


DATA_FILE = annual_trip_file(YEAR)
rides_data = pd.read_parquet(DATA_FILE)


In [54]:
rides_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
0,2.0,CLASSIC,2025-01-14 18:27:00,51.497304,-0.022793,200109.0,"Alpha Grove, Millwall",8.042234,2025-01-14 18:03:00,Tuesday,51.508622,-0.065006,200090,"Pennington Street, Wapping",3.181329,23m 44s,23.734667,145660049
1,2.0,CLASSIC,2025-01-14 21:07:00,51.505703,-0.027773,300228.0,"Westferry Circus, Canary Wharf",9.712075,2025-01-14 21:00:00,Tuesday,51.497304,-0.022793,200109,"Alpha Grove, Millwall",0.995509,6m 9s,6.150133,145665363
2,2.0,CLASSIC,2025-01-14 21:52:00,51.501513,-0.123823,300249.0,"Westminster Pier, Westminster",10.431536,2025-01-14 21:14:00,Tuesday,51.505703,-0.027773,300228,"Westferry Circus, Canary Wharf",6.664447,38m 19s,38.332500,145665536
3,2.0,CLASSIC,2025-01-14 22:36:00,51.497304,-0.022793,200109.0,"Alpha Grove, Millwall",9.840540,2025-01-14 21:54:00,Tuesday,51.501513,-0.123823,300249,"Westminster Pier, Westminster",7.009102,42m 44s,42.736083,145666019
4,2.0,CLASSIC,2025-01-17 14:40:00,51.503400,-0.020242,200163.0,"Jubilee Plaza, Canary Wharf",8.874631,2025-01-17 14:35:00,Friday,51.497304,-0.022793,200109,"Alpha Grove, Millwall",0.700462,4m 44s,4.735717,145727708


In [55]:
all_bikes_id = rides_data["bike_id"].unique()

In [56]:
len(all_bikes_id)

13676

In [57]:
# data_preparation_eksperyment.ipynb sorts trips before writing the annual Parquet file.
def validate_trip_order(trips, batch_size=250_000, data_file=DATA_FILE):
    """Check the saved sort order using small, overlapping batches."""

    for date_column in ["start_date", "end_date"]:
        trips[date_column] = pd.to_datetime(
            trips[date_column], format="ISO8601", errors="raise"
        )

    if trips[["bike_id", "start_date", "end_date"]].isna().any().any():
        raise ValueError("Continuity detection requires bike IDs and valid trip dates.")

    for start in range(1, len(trips), batch_size):
        block = trips.iloc[start - 1:start + batch_size]
        bike = block["bike_id"].to_numpy()
        begin = block["start_date"].to_numpy()
        end = block["end_date"].to_numpy()
        decreasing = (
            (bike[1:] < bike[:-1])
            | ((bike[1:] == bike[:-1]) & (
                (begin[1:] < begin[:-1])
                | ((begin[1:] == begin[:-1]) & (end[1:] < end[:-1]))
            ))
        )
        if decreasing.any():
            raise ValueError(
                "Trips are not sorted by bike_id, start_date, end_date. "
                f"Regenerate {data_file.name} with data_preparation_eksperyment.ipynb."
            )


validate_trip_order(rides_data)

# ---------- Previous trip information for the same bike ----------
rides_data["previous_end_station_id"] = (
    rides_data
    .groupby("bike_id")["end_station_id"]
    .shift(1)
)

rides_data["previous_end_date"] = (
    rides_data
    .groupby("bike_id")["end_date"]
    .shift(1)
)
rides_data["previous_end_station_name"] = (
    rides_data
    .groupby("bike_id")["end_station_name"]
    .shift(1)
)

In [58]:
# Complete the first trip of each bike with its most recent earlier trip.
def add_previous_year_trips(trips):
    first_bikes = trips.loc[~trips["bike_id"].duplicated(), "bike_id"]
    year_pattern = "[0-9]" * 4
    prepared_years = {
        int(path.stem[-4:])
        for path in DATA_DIR.glob(f"final_trip_data_{year_pattern}.parquet")
    }
    source_years = {
        int(path.name) for path in (PROJECT_ROOT / "raw_data").glob(year_pattern) if path.is_dir()
    }
    earlier_years = sorted((y for y in prepared_years | source_years if y < YEAR), reverse=True)

    for year in earlier_years:
        if first_bikes.empty:
            break
        path = annual_trip_file(year)
        previous = pd.read_parquet(
            path, columns=["bike_id", "start_date", "end_date", "end_station_id", "end_station_name"]
        )
        validate_trip_order(previous, data_file=path)
        # The last row in the saved sort order is also the predecessor used by shift(1).
        last = previous.drop_duplicates("bike_id", keep="last").set_index("bike_id")
        del previous
        matched = first_bikes[first_bikes.isin(last.index)]
        if matched.empty:
            continue
        last = last.loc[matched.to_numpy()]

        # An earlier station name may be absent from this year's category dictionary.
        names = trips["previous_end_station_name"]
        if isinstance(names.dtype, pd.CategoricalDtype):
            extra_names = pd.Index(last["end_station_name"].dropna().unique()).difference(names.cat.categories)
            if len(extra_names):
                trips["previous_end_station_name"] = names.cat.add_categories(extra_names)

        for column in ["end_station_id", "end_date", "end_station_name"]:
            trips.loc[matched.index, f"previous_{column}"] = last[column].to_numpy()
        first_bikes = first_bikes.drop(matched.index)


add_previous_year_trips(rides_data)


In [59]:
rides_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,...,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id,previous_end_station_id,previous_end_date,previous_end_station_name
0,2.0,CLASSIC,2025-01-14 18:27:00,51.497304,-0.022793,200109.0,"Alpha Grove, Millwall",8.042234,2025-01-14 18:03:00,Tuesday,...,-0.065006,200090,"Pennington Street, Wapping",3.181329,23m 44s,23.734667,145660049,999.0,2024-12-20 08:20:00,"Queen Street 1, Bank"
1,2.0,CLASSIC,2025-01-14 21:07:00,51.505703,-0.027773,300228.0,"Westferry Circus, Canary Wharf",9.712075,2025-01-14 21:00:00,Tuesday,...,-0.022793,200109,"Alpha Grove, Millwall",0.995509,6m 9s,6.150133,145665363,200109.0,2025-01-14 18:27:00,"Alpha Grove, Millwall"
2,2.0,CLASSIC,2025-01-14 21:52:00,51.501513,-0.123823,300249.0,"Westminster Pier, Westminster",10.431536,2025-01-14 21:14:00,Tuesday,...,-0.027773,300228,"Westferry Circus, Canary Wharf",6.664447,38m 19s,38.332500,145665536,300228.0,2025-01-14 21:07:00,"Westferry Circus, Canary Wharf"
3,2.0,CLASSIC,2025-01-14 22:36:00,51.497304,-0.022793,200109.0,"Alpha Grove, Millwall",9.840540,2025-01-14 21:54:00,Tuesday,...,-0.123823,300249,"Westminster Pier, Westminster",7.009102,42m 44s,42.736083,145666019,300249.0,2025-01-14 21:52:00,"Westminster Pier, Westminster"
4,2.0,CLASSIC,2025-01-17 14:40:00,51.503400,-0.020242,200163.0,"Jubilee Plaza, Canary Wharf",8.874631,2025-01-17 14:35:00,Friday,...,-0.022793,200109,"Alpha Grove, Millwall",0.700462,4m 44s,4.735717,145727708,200109.0,2025-01-14 22:36:00,"Alpha Grove, Millwall"


# ---------- Detecting discontinuities ----------


In [60]:

discontinuity_mask = (
    rides_data["previous_end_station_id"].notna()
    & (rides_data["previous_end_station_id"] != rides_data["start_station_id"])
)

bike_discontinuities = rides_data[discontinuity_mask]

In [61]:
bike_discontinuities["time_gap"] = (
    bike_discontinuities["start_date"] - bike_discontinuities["previous_end_date"]
)

In [62]:
bike_discontinuities["time_gap_minutes"] = (
    bike_discontinuities["time_gap"].dt.total_seconds() / 60
)

In [63]:
bike_discontinuities_table = bike_discontinuities[
    [
        "bike_id",
        "bike_model",
        "previous_end_date",
        "previous_end_station_id",
        "previous_end_station_name",
        "start_date",
        "start_station_id",
        "start_station_name",
        "end_date",
        "end_station_id",
        "end_station_name",
        "time_gap",
        "time_gap_minutes",
        "trip_id"
    ]
]

In [64]:
bike_discontinuities_table.head(20)

,bike_id,bike_model,previous_end_date,previous_end_station_id,previous_end_station_name,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,time_gap,time_gap_minutes,trip_id
0,2.0,CLASSIC,2024-12-20 08:20:00,999.0,"Queen Street 1, Bank",2025-01-14 18:03:00,200090,"Pennington Street, Wapping",2025-01-14 18:27:00,200109.0,"Alpha Grove, Millwall",25 days 09:43:00,36583.0,145660049
5,2.0,CLASSIC,2025-01-17 14:40:00,200163.0,"Jubilee Plaza, Canary Wharf",2025-01-17 18:43:00,200038,"Ackroyd Drive, Bow",2025-01-17 19:03:00,200036.0,"Bell Lane, Liverpool Street",0 days 04:03:00,243.0,145734396
11,2.0,CLASSIC,2025-01-19 11:23:00,3429.0,"Abbey Orchard Street, Westminster",2025-01-20 20:44:00,300096,"Tallis Street, Temple",2025-01-20 21:02:00,200115.0,"Flamborough Street, Limehouse",1 days 09:21:00,2001.0,145790358
13,2.0,CLASSIC,2025-01-21 08:57:00,300096.0,"Tallis Street, Temple",2025-01-22 07:48:00,1023,"River Street , Clerkenwell",2025-01-22 08:01:00,2688.0,"Clifford Street, Mayfair",0 days 22:51:00,1371.0,145819816
21,2.0,CLASSIC,2025-01-24 06:23:00,1153.0,"Pall Mall East, West End",2025-01-25 12:15:00,300079,"London Street, Paddington",2025-01-25 12:41:00,3469.0,"Cadogan Place, Knightsbridge",1 days 05:52:00,1792.0,145892622
25,2.0,CLASSIC,2025-01-29 08:18:00,1067.0,"St. James's Square, St. James's",2025-01-30 09:51:00,1227,"Devonshire Terrace, Bayswater",2025-01-30 10:02:00,3493.0,"South Audley Street, Mayfair",1 days 01:33:00,1533.0,145991041
28,2.0,CLASSIC,2025-01-31 10:49:00,3493.0,"South Audley Street, Mayfair",2025-01-31 18:24:00,3482,"Beaumont Street, Marylebone",2025-01-31 18:35:00,200118.0,"Albert Street, Camden Town",0 days 07:35:00,455.0,146023532
33,2.0,CLASSIC,2025-02-03 09:57:00,200226.0,"Kingsway Southbound, Strand",2025-02-22 13:24:00,22172,"Bromley High Street, Bow",2025-02-22 13:48:00,200064.0,"Regent's Row , Haggerston",19 days 03:27:00,27567.0,146467189
43,2.0,CLASSIC,2025-03-04 17:19:00,991.0,"Crosswall, Tower",2025-03-07 10:10:00,200251,"Bow Road Station, Bow",2025-03-07 10:23:00,300049.0,"Aylward Street, Stepney",2 days 16:51:00,3891.0,146772693
60,2.0,CLASSIC,2025-03-22 00:18:00,1153.0,"Pall Mall East, West End",2025-03-23 11:39:00,300201,"Coram Street, Bloomsbury",2025-03-23 11:45:00,1011.0,"Argyle Street, Kings Cross",1 days 11:21:00,2121.0,147177658


In [65]:
print(f"Number of discontinuities: {len(bike_discontinuities_table)}")

Number of discontinuities: 499278


In [66]:
discontinuity_percent = (
    len(bike_discontinuities_table) / len(rides_data) * 100
)

print(f"Percentage of discontinuous trips: {discontinuity_percent:.3f}%")

Percentage of discontinuous trips: 5.606%


In [67]:
rides_data_continuous = rides_data[~discontinuity_mask]

In [68]:
rides_data_continuous.to_parquet(DATA_DIR /
    f"rides_data_without_discontinuities_{YEAR}.parquet",
    index=False
)